In [1]:
import pandas as pd


In [2]:
excel_file = pd.ExcelFile('Ardent_Mills_Data.xlsx')
all_sheets= {}

In [3]:
for sheet_name in excel_file.sheet_names:
    all_sheets[sheet_name] = pd.read_excel(excel_file, sheet_name=sheet_name)



In [4]:
pack = all_sheets['Pack']
mill = all_sheets['Mill']
sales = all_sheets['Sales']
workorder = all_sheets['WorkOrder']
bincleaning = all_sheets['BinCleaning']
fills = all_sheets['Fills']

In [5]:
sales.dtypes

SITE_SHORT_NAME                   object
ORDER_NO                          object
CUSTOMER_NM                       object
SHIP_DATE                 datetime64[ns]
ITEM_CLASS_DESC                   object
ITEM_NUM                          object
ITEM_DESC                         object
ORDER_STATUS_INDICATOR            object
INVOICE_CWTS                     float64
dtype: object

In [6]:
# 1. Split site column
sales[['SITE_NAME', 'SITE_ID']] = sales['SITE_SHORT_NAME'].str.split('-', expand=True)
sales.drop(columns='SITE_SHORT_NAME', inplace=True)

# 2. Reorder columns
sales = sales[['SITE_ID', 'SITE_NAME', 'ORDER_NO', 'CUSTOMER_NM', 'SHIP_DATE',
               'ITEM_CLASS_DESC', 'ITEM_NUM', 'ITEM_DESC', 'ORDER_STATUS_INDICATOR', 'INVOICE_CWTS']]

# 3. Safe strip — handles mixed int/string object columns
sales = sales.apply(lambda col: col.apply(lambda x: x.strip() if isinstance(x, str) else x) if col.dtype == 'object' else col)

# 4. Convert integer ITEM_NUM and ORDER_NO to string
sales['ITEM_NUM'] = sales['ITEM_NUM'].apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and pd.notna(x) else x)
sales['ORDER_NO'] = sales['ORDER_NO'].apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and pd.notna(x) else x)

# 5. Remove dash from ITEM_NUM
sales['ITEM_NUM'] = sales['ITEM_NUM'].str.replace('-', '', regex=False)

# 6. Drop duplicates
sales = sales.drop_duplicates()

# Verify
print(sales['ITEM_NUM'].nunique())  # ~197
print(sales['ORDER_NO'].nunique())  # ~1321
print(sales.head())

197
1325
  SITE_ID SITE_NAME  ORDER_NO        CUSTOMER_NM  SHIP_DATE   ITEM_CLASS_DESC  \
0    1001    ALBANY  82378825  ORLANDO FOODS INC 2017-03-10        HARD FLOUR   
1    1025     OGDEN  M0060739   LARKIN CATTLE CO 2017-03-11  WHEAT BYPRODUCTS   
2    1025     OGDEN  M0291253   LARKIN CATTLE CO 2017-03-20  WHEAT BYPRODUCTS   
3    1025     OGDEN  M0289716   LARKIN CATTLE CO 2017-03-13  WHEAT BYPRODUCTS   
4    1025     OGDEN  M0291254   LARKIN CATTLE CO 2017-03-18  WHEAT BYPRODUCTS   

   ITEM_NUM                   ITEM_DESC ORDER_STATUS_INDICATOR  INVOICE_CWTS  
0   5122073        ROSE HG FLR 50LB-RK7               INVOICED         450.0  
1  06000001  WHEAT MIDDLINGS LOOSE-BULK               INVOICED         600.0  
2  06000001  WHEAT MIDDLINGS LOOSE-BULK               INVOICED         578.4  
3  06000001  WHEAT MIDDLINGS LOOSE-BULK               INVOICED         579.0  
4  06000001  WHEAT MIDDLINGS LOOSE-BULK               INVOICED         659.4  


In [7]:
sales.dtypes

SITE_ID                           object
SITE_NAME                         object
ORDER_NO                          object
CUSTOMER_NM                       object
SHIP_DATE                 datetime64[ns]
ITEM_CLASS_DESC                   object
ITEM_NUM                          object
ITEM_DESC                         object
ORDER_STATUS_INDICATOR            object
INVOICE_CWTS                     float64
dtype: object

In [8]:
sales.isnull().sum()

SITE_ID                   0
SITE_NAME                 0
ORDER_NO                  0
CUSTOMER_NM               0
SHIP_DATE                 0
ITEM_CLASS_DESC           0
ITEM_NUM                  0
ITEM_DESC                 0
ORDER_STATUS_INDICATOR    0
INVOICE_CWTS              0
dtype: int64

In [9]:
sales['ORDER_STATUS_INDICATOR'].value_counts()

ORDER_STATUS_INDICATOR
INVOICED              1465
SHIPPED NOT BILLED     125
TRUE-RETURN              9
RETURNED                 7
Name: count, dtype: int64

In [10]:
sales[sales['INVOICE_CWTS'] < 0]

,SITE_ID,SITE_NAME,ORDER_NO,CUSTOMER_NM,SHIP_DATE,ITEM_CLASS_DESC,ITEM_NUM,ITEM_DESC,ORDER_STATUS_INDICATOR,INVOICE_CWTS
48,1025,OGDEN,82390689,TREEHOUSE PRIVATE BRANDS,2017-03-10,WHITE WHOLE WHEAT,5164114,ARDENT SPR WHITE WW FLR 50LB-AA,RETURNED,-50.00
302,1001,ALBANY,82391928,DOMINOS PIZZA LLC - ANN ARBOR MI,2017-03-19,HARD FLOUR,5138429,DOMINOS BUBBA FLR BULK-RA-1817,RETURNED,-519.00
471,1001,ALBANY,82390480,THE J M SMUCKER - CLR/MIDDS,2017-03-17,HARD FLOUR,5100214,2ND CLEAR FLR BULK-AA-R51604(FEED),RETURNED,-506.00
529,1001,ALBANY,M0061295,CFN 1001 SHIPPENSBURG,2017-03-21,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,TRUE-RETURN,-1167.83
582,1004,AYER,M0060890,KENT NUTRITION GROUP INCO,2017-03-16,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,TRUE-RETURN,-456.00
915,1001,ALBANY,M0289739,LAND O'LAKES FEED LLC,2017-03-14,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,TRUE-RETURN,-483.82
1264,1001,ALBANY,M0289738,FEED INGREDIENT TRADING CORP,2017-03-13,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,TRUE-RETURN,-497.03
1271,1001,ALBANY,M0288695,FEED INGREDIENT TRADING CORP,2017-03-13,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,TRUE-RETURN,-578.87
1406,1004,AYER,82387386,NORTHEAST FOODS INCORPORA,2017-03-13,HARD FLOUR,5119327,RELIANCE FLR BULK-RG,RETURNED,-513.50
1495,1001,ALBANY,M0061260,GRAIN ST LAURENT,2017-03-17,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,TRUE-RETURN,-497.62


In [11]:
print(sales['SHIP_DATE'].min())
print(sales['SHIP_DATE'].max())
# Should all be within March 2017
# Flag any outliers


2017-03-10 00:00:00
2017-03-23 00:00:00


In [12]:
sales['ITEM_CLASS_DESC'].value_counts()
# Should have exactly 7 categories:
# HARD FLOUR | SOFT FLOUR | WHEAT BYPRODUCTS | ORGANIC
# BRAN/GERM/NUFIBER | WHITE WHOLE WHEAT | RED WHOLE WHEAT


ITEM_CLASS_DESC
HARD FLOUR           1168
WHEAT BYPRODUCTS      274
SOFT FLOUR            127
WHITE WHOLE WHEAT      19
RED WHOLE WHEAT         9
ORGANIC                 6
BRAN/GERM/NUFIBER       3
Name: count, dtype: int64

In [13]:
sales.dtypes

SITE_ID                           object
SITE_NAME                         object
ORDER_NO                          object
CUSTOMER_NM                       object
SHIP_DATE                 datetime64[ns]
ITEM_CLASS_DESC                   object
ITEM_NUM                          object
ITEM_DESC                         object
ORDER_STATUS_INDICATOR            object
INVOICE_CWTS                     float64
dtype: object

In [14]:
sales['CUSTOMER_NM'].nunique()
# Should be 145 after stripping spaces


145

In [15]:
sales['ITEM_CLASS_DESC'].value_counts()
# Should have exactly 7 categories:
# HARD FLOUR | SOFT FLOUR | WHEAT BYPRODUCTS | ORGANIC
# BRAN/GERM/NUFIBER | WHITE WHOLE WHEAT | RED WHOLE WHEAT


ITEM_CLASS_DESC
HARD FLOUR           1168
WHEAT BYPRODUCTS      274
SOFT FLOUR            127
WHITE WHOLE WHEAT      19
RED WHOLE WHEAT         9
ORGANIC                 6
BRAN/GERM/NUFIBER       3
Name: count, dtype: int64

In [16]:
# fills['exception_cwts'].isnull().sum()   # ~384 nulls — expected
# fills['exception_flag'].value_counts()   # Exception: 107 | No Exception: 384


In [17]:
sales.to_csv('sales_cleaned.csv',header=True,index=False)

In [18]:
sales_cleaned = sales.copy()

In [19]:
sales_cleaned

,SITE_ID,SITE_NAME,ORDER_NO,CUSTOMER_NM,SHIP_DATE,ITEM_CLASS_DESC,ITEM_NUM,ITEM_DESC,ORDER_STATUS_INDICATOR,INVOICE_CWTS
0,1001,ALBANY,82378825,ORLANDO FOODS INC,2017-03-10,HARD FLOUR,5122073,ROSE HG FLR 50LB-RK7,INVOICED,450.00
1,1025,OGDEN,M0060739,LARKIN CATTLE CO,2017-03-11,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,600.00
2,1025,OGDEN,M0291253,LARKIN CATTLE CO,2017-03-20,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,578.40
3,1025,OGDEN,M0289716,LARKIN CATTLE CO,2017-03-13,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,579.00
4,1025,OGDEN,M0291254,LARKIN CATTLE CO,2017-03-18,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,659.40
...,...,...,...,...,...,...,...,...,...,...
1605,1001,ALBANY,M0061332,APEX LLC,2017-03-21,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,496.84
1606,1001,ALBANY,M0061334,APEX LLC,2017-03-23,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,498.07
1607,1001,ALBANY,M0061529,APEX LLC,2017-03-22,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,498.66
1608,1001,ALBANY,M0061333,APEX LLC,2017-03-22,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,497.45


In [20]:
mill

,SITE_SHORT_NAME,MILLDATE,UNIT,ProductionMix,Calculated Downtime,MinRun,NODemand Downtime,Mill OEE
0,AYER-1004,2017-03-15,A,H54,4.971176,327.0,0.0,0.984798
1,AYER-1004,2017-03-15,A,H50,9.427941,1113.0,0.0,0.991529
2,AYER-1004,2017-03-15,B,M54,-5.563448,1440.0,0.0,1.003864
3,AYER-1004,2017-03-14,A,H50,1.900588,345.0,0.0,0.994491
4,AYER-1004,2017-03-14,B,M54,-28.433793,345.0,0.0,1.082417
...,...,...,...,...,...,...,...,...
362,OGDEN-1025,2017-03-21,A,AW42 CA46,4.430769,165.0,0.0,0.973147
363,OGDEN-1025,2017-03-21,B,E45,11.274545,465.0,0.0,0.975754
364,OGDEN-1025,2017-03-21,B,D58,0.256364,75.0,0.0,0.996582
365,OGDEN-1025,2017-03-21,B,M56,-1.053636,10.0,0.0,1.105364


In [21]:
sales

,SITE_ID,SITE_NAME,ORDER_NO,CUSTOMER_NM,SHIP_DATE,ITEM_CLASS_DESC,ITEM_NUM,ITEM_DESC,ORDER_STATUS_INDICATOR,INVOICE_CWTS
0,1001,ALBANY,82378825,ORLANDO FOODS INC,2017-03-10,HARD FLOUR,5122073,ROSE HG FLR 50LB-RK7,INVOICED,450.00
1,1025,OGDEN,M0060739,LARKIN CATTLE CO,2017-03-11,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,600.00
2,1025,OGDEN,M0291253,LARKIN CATTLE CO,2017-03-20,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,578.40
3,1025,OGDEN,M0289716,LARKIN CATTLE CO,2017-03-13,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,579.00
4,1025,OGDEN,M0291254,LARKIN CATTLE CO,2017-03-18,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,659.40
...,...,...,...,...,...,...,...,...,...,...
1605,1001,ALBANY,M0061332,APEX LLC,2017-03-21,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,496.84
1606,1001,ALBANY,M0061334,APEX LLC,2017-03-23,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,498.07
1607,1001,ALBANY,M0061529,APEX LLC,2017-03-22,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,498.66
1608,1001,ALBANY,M0061333,APEX LLC,2017-03-22,WHEAT BYPRODUCTS,06000001,WHEAT MIDDLINGS LOOSE-BULK,INVOICED,497.45
